# Cell line preprocessing

Notebook counterpart of `scripts/02_cell_line_pp.py`. The steps are the
same: load the large SCP542 table in sparse chunks, map genes to the
shared table, keep colorectal cell lines, drop lines with fewer than 50
cells, attach MSI and batch metadata, and set categorical dtypes.

The notebook does not save the processed artifact; the script does.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / "config" / "config.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import cell_line_preprocessing as pp
from src.utils import load_config

config = load_config()
raw = config["paths"]["raw_data"]
processed = config["paths"]["processed"]

cell_line_path = ROOT / raw["cell_line"]
gene_path = ROOT / processed["genes_combined"]

In [2]:
adata = pp.load_cell_line_csv_as_adata(
    cell_line_path, cell_line_contains="LARGE_INTESTINE"
)
print(f"loaded: {adata.n_obs:,} cells x {adata.n_vars:,} genes")
print(adata.obs["cell_line"].value_counts())

    Loading 2256 selected cell columns in sparse chunks...
    Processed 25 gene chunks...
    Processed 50 gene chunks...
loaded: 2,256 cells x 30,314 genes
cell_line
SNUC4_LARGE_INTESTINE      471
CL34_LARGE_INTESTINE       418
LS180_LARGE_INTESTINE      362
HT55_LARGE_INTESTINE       211
LS1034_LARGE_INTESTINE     153
RCM1_LARGE_INTESTINE       131
CCK81_LARGE_INTESTINE      127
OUMS23_LARGE_INTESTINE     107
HCC56_LARGE_INTESTINE      102
NCIH747_LARGE_INTESTINE     80
SNU81_LARGE_INTESTINE       49
SW1116_LARGE_INTESTINE      23
SW1463_LARGE_INTESTINE      22
Name: count, dtype: int64


In [3]:
adata = pp.subset_to_gene_intersection_with_gene_id(adata, gene_path)
adata = pp.rename_cell_lines(adata)
adata = pp.remove_cell_lines_lower_50_cells(adata)
adata = pp.attach_extra_metadata(adata)
adata = pp.cols_to_cat(adata)
print(f"retained: {adata.n_obs:,} cells x {adata.n_vars:,} genes")
print(f"cell lines: {adata.obs['cell_line'].nunique()}")
adata.obs[["cell_line", "pool_id", "batch", "mmr_status"]].head()

    Sanity check: All intersection genes in cell line data? True
retained: 2,162 cells x 28,617 genes
cell lines: 10


,cell_line,pool_id,batch,mmr_status
AATCCAGGTTTAGGAA-1-18,NCIH747,18,18,MMRp
ACATCAGTCATGGTCA-1-18,NCIH747,18,18,MMRp
ATCATGGTCGCATGAT-1-18,NCIH747,18,18,MMRp
ATCCGAAAGCGCTTAT-1-18,NCIH747,18,18,MMRp
ATCTGCCCACAGATTC-1-18,NCIH747,18,18,MMRp


The script writes `data/processed/preprocessed_cell_line.h5ad`; the
notebook stops before the write so it can be run without changing the
pipeline artifacts.